# ⚡ Performance & Otimização de Computação Distribuída no Databricks

Este notebook aborda as técnicas essenciais de otimização no Apache Spark e Delta Lake:
- Compactação de pequenos arquivos com `OPTIMIZE`
- Ordenação multidimensional com `Z-ORDER BY`
- Propriedades de auto-otimização (`delta.autoOptimize`)
- Estratégias de particionamento e colunas geradas
- Broadcast Hash Join Hints (`/*+ BROADCAST */`)
- Estatísticas de custo com `ANALYZE TABLE`

In [ ]:
# 1. Criação de Tabelas Gerenciadas para Simulação de Performance
spark.sql("""
CREATE OR REPLACE TABLE clientes (
    cliente_id BIGINT,
    nome STRING,
    email STRING,
    data_cadastro DATE,
    ativo BOOLEAN
) USING DELTA PARTITIONED BY (data_cadastro);
""")

spark.sql("""
CREATE OR REPLACE TABLE produtos (
    produto_id BIGINT,
    nome_produto STRING,
    categoria STRING,
    preco DOUBLE
) USING DELTA;
""")

spark.sql("""
CREATE OR REPLACE TABLE vendas (
    venda_id BIGINT,
    cliente_id BIGINT,
    produto_id BIGINT,
    quantidade INT,
    valor_total DOUBLE,
    data_venda DATE,
    status STRING
) USING DELTA;
""")

print("✅ Tabelas Delta criadas com sucesso!")

In [ ]:
# 2. Compactação de Arquivos Pequenos e Z-ORDER
# O Z-ORDER reduz drasticamente o tempo de consulta por co-localizar dados relacionados
spark.sql("""
OPTIMIZE vendas
ZORDER BY (cliente_id, produto_id);
""")

# Habilitando auto-otimização na tabela
spark.sql("""
ALTER TABLE vendas SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
""")
print("✅ Tabela de vendas otimizada com Z-ORDER e AutoOptimize!")

In [ ]:
# 3. Broadcast Hash Join vs Shuffle Join
# O Broadcast envia a tabela dimensional pequena para todos os executores, eliminando o Shuffle pela rede
df_join_otimizado = spark.sql("""
SELECT /*+ BROADCAST(p) */
    v.venda_id,
    v.data_venda,
    p.nome_produto,
    p.categoria,
    v.valor_total
FROM vendas v
JOIN produtos p ON v.produto_id = p.produto_id
""")

# Exibe o plano de execução físico confirmando o BroadcastExchange
df_join_otimizado.explain(True)

In [ ]:
# 4. Coleta de Estatísticas para o Cost-Based Optimizer (CBO)
spark.sql("ANALYZE TABLE vendas COMPUTE STATISTICS FOR COLUMNS cliente_id, produto_id, data_venda;")
display(spark.sql("DESCRIBE EXTENDED vendas"))